# 27 · Base-rate initialised heads against the frequency baseline

Design log
- Notebook 26: converged heads had higher top-1 endorsement than the frequency baseline but higher log-loss, on validation and on the calibration partition.
- Amendment: TrainSpec.bias_init = "prior" zeroes the output layer weights and sets each output bias to the fit-partition base-rate log-odds, so the untrained head reproduces the frequency baseline exactly. The value "default" keeps the previous initialisation.
- The amendment changes the package source hash. Notebook 05 runs therefore show as incompatible with the principal grid and are re-run under the amended code before notebook 08. Notebook 05 and 26 results are unchanged.
- Same learning-rate grid and epoch cap as notebook 26. Runs are written under runs/foundation_prior_init/.
- The configuration is selected by validation log-loss and reported once on the calibration partition. Test records are not read.

In [ ]:
from pathlib import Path
import os, sys, json
ON_COLAB = "google.colab" in sys.modules or bool(os.environ.get("COLAB_RELEASE_TAG"))
if ON_COLAB and not Path("/content/drive/MyDrive").exists():
    from google.colab import drive
    drive.mount("/content/drive")
ROOT = Path(os.environ.get("ONCOPLATE_DRIVE_ROOT", "/content/drive/MyDrive/OncoPlate_Research"))
pointer = ROOT / ".oncoplate_install.json"
preferred = json.loads(pointer.read_text())["repository_path"] if pointer.exists() else str(ROOT / "oncoplate-research")
REPO = Path(os.environ.get("ONCOPLATE_REPO", preferred))
if not (REPO / "src/oncoplate").exists():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "src/oncoplate").exists(): REPO = candidate; break
assert (REPO / "src/oncoplate").exists(), "Run the supplied installer notebook or set ONCOPLATE_REPO to the extracted repository."
sys.path.insert(0, str(REPO / "src"))
from oncoplate.config import load_config, paths, initialize
from oncoplate.io import read_json, write_json, read_table, write_table, read_jsonl, write_jsonl, utcnow
cfg = load_config(REPO, root=ROOT, mode=os.environ.get("ONCOPLATE_MODE", "research"))
p = paths(cfg)
print("Dataset:", cfg["study"]["dataset"], "| Mode:", cfg["mode"], "| Persistent root:", cfg["root"])


## 1. Frequency baseline, helpers and initialisation check

In [ ]:
import numpy as np, pandas as pd, torch
from dataclasses import replace
from oncoplate.pipeline import frequency_baseline, load_study, spec_from_cfg, prediction_arrays
from oncoplate.training import train_predictor, predict_run, Head, prior_initialise
from oncoplate.targets import align_targets
from oncoplate.calibration import calibration_metrics, probabilities
from oncoplate.statistics import analysis_weights

HEADS = ("independent", "joint")
STUDY = {h: load_study(cfg, h, stage_images=True) for h in HEADS}
PREVALENCE = {h: frequency_baseline(cfg, h, "validation")["probabilities"][0] for h in HEADS}

def evaluate(prob, y, mask, ids, head):
    m = calibration_metrics(prob, y, mask)
    sub = STUDY[head][0].set_index("record_id").loc[ids].reset_index()
    w = analysis_weights(sub, {"meal": 1.0})
    rows = np.arange(len(prob)); j = prob.argmax(1)
    observed = mask[rows, j] > 0; den = w[observed].sum()
    top1 = float(np.sum(w[observed] * y[rows, j][observed]) / den) if den else float("nan")
    return {"log_loss": m["log_loss"], "ece": m["ece"], "top1_endorsement": top1}

def frequency_on(ids, y, mask, head):
    # Scored on exactly the records, targets and masks the image model is scored on.
    return evaluate(np.broadcast_to(PREVALENCE[head], y.shape).copy(), y, mask, ids, head)

# The amended initialisation must reproduce the frequency baseline before any training.
for head in HEADS:
    records, targets = STUDY[head]
    fy, fm = align_targets(targets, records.loc[records.split == "fit", "record_id"])
    probe = Head(8, fy.shape[1]); prior_initialise(probe, fy, fm)
    with torch.no_grad(): out = torch.sigmoid(probe(torch.randn(4, 8))).numpy()
    assert np.allclose(out, np.clip(PREVALENCE[head], 1e-4, 1 - 1e-4)[None, :], atol=1e-5), head
print("untrained base-rate heads reproduce the frequency baseline")

## 2. Learning-rate grid with base-rate initialisation

In [ ]:
LEARNING_RATES = (0.001, 0.003, 0.01)
EPOCHS = 400
OUT = p["runs"] / "foundation_prior_init"
rows = []
for head in HEADS:
    records, targets = STUDY[head]
    base = spec_from_cfg(cfg, "resnet50", "frozen", head, 0)
    for lr in LEARNING_RATES:
        spec = replace(base, lr=lr, epochs=EPOCHS, bias_init="prior")
        run_dir = OUT / f"{spec.run_id}_prior_lr{lr:g}_e{EPOCHS}"
        train_predictor(records, targets, spec, run_dir, p["features"])
        done = read_json(run_dir / "complete.json")
        rows.append({"head": head, "lr": lr, "run_dir": str(run_dir),
                     "epochs_completed": int(done["epochs_completed"]),
                     "early_stopped": int(done["epochs_completed"]) < EPOCHS,
                     "best_validation_bce": float(done["best_validation_loss"])})
grid = pd.DataFrame(rows)
write_table(p["reports"] / "foundation_prior_init_grid.csv", grid)
grid

## 3. Validation comparison on identical records

In [ ]:
PREVIOUS = p["runs"] / "foundation_convergence"
table = []
for head in HEADS:
    ref = prediction_arrays(p["runs"] / f"resnet50_frozen_{head}_s0" / "validation_predictions.npz")
    ids, y, mask = ref["ids"], ref["y"], ref["mask"]
    freq_row = frequency_on(ids, y, mask, head)
    saved = read_json(p["reports"] / f"frequency_{head}_validation_metrics.json")["log_loss"]
    assert np.isclose(freq_row["log_loss"], saved, rtol=1e-5), "Frequency baseline does not reproduce notebook 05"
    table.append({"head": head, "init": "none", "model": "frequency baseline (no image)", **freq_row})
    for lr in LEARNING_RATES:
        z = prediction_arrays(PREVIOUS / f"resnet50_frozen_{head}_s0_lr{lr:g}_e{EPOCHS}" / "validation_predictions.npz")
        assert list(z["ids"]) == list(ids), "Validation record sets differ"
        table.append({"head": head, "init": "default (notebook 26)", "model": f"lr {lr:g}",
                      **evaluate(probabilities(z["logits"]), y, mask, ids, head)})
    for r in grid[grid["head"] == head].itertuples():
        z = prediction_arrays(Path(r.run_dir) / "validation_predictions.npz")
        assert list(z["ids"]) == list(ids), "Validation record sets differ"
        table.append({"head": head, "init": "base rate", "model": f"lr {r.lr:g}, stopped at {r.epochs_completed} of {EPOCHS}",
                      **evaluate(probabilities(z["logits"]), y, mask, ids, head)})
validation = pd.DataFrame(table)
write_table(p["reports"] / "foundation_prior_init_validation.csv", validation)
validation

## 4. Selected configuration on the calibration partition

In [ ]:
chosen = grid.sort_values(["head", "best_validation_bce", "lr"]).groupby("head").head(1)
confirm, verdict = [], {}
for r in chosen.itertuples():
    records, targets = STUDY[r.head]
    pred = predict_run(Path(r.run_dir), records[records.split.eq("calibration")], targets, p["features"])
    ids, y, mask = pred["ids"], pred["y"], pred["mask"]
    freq_row = frequency_on(ids, y, mask, r.head)
    model_row = evaluate(probabilities(pred["logits"]), y, mask, ids, r.head)
    confirm += [{"head": r.head, "model": "frequency baseline (no image)", **freq_row},
                {"head": r.head, "model": f"lr {r.lr:g}, stopped at {r.epochs_completed} of {EPOCHS}", **model_row}]
    verdict[r.head] = {"selected_lr": float(r.lr), "epochs_completed": int(r.epochs_completed),
                       "early_stopped": bool(r.early_stopped),
                       "lower_log_loss_than_frequency": bool(model_row["log_loss"] < freq_row["log_loss"]),
                       "higher_top1_endorsement_than_frequency": bool(model_row["top1_endorsement"] > freq_row["top1_endorsement"])}
calibration_partition = pd.DataFrame(confirm)
write_table(p["reports"] / "foundation_prior_init_calibration_partition.csv", calibration_partition)
write_json(p["reports"] / "foundation_prior_init_summary.json", verdict)
display(calibration_partition)
print(json.dumps(verdict, indent=2))